In [1]:
import pandas as pd
import seaborn as sns
import numpy as np

In [2]:
transaction_pd = pd.read_csv('C:/Users/Playdata/Downloads/h-and-m-personalized-fashion-recommendations/transactions_train.csv')
customer_df = pd.read_csv('C:/Users/Playdata/Downloads/h-and-m-personalized-fashion-recommendations/customers.csv')
art_df = pd.read_csv('C:/Users/Playdata/Downloads/h-and-m-personalized-fashion-recommendations/articles.csv')

In [3]:
print(len(transaction_pd))
print(len(customer_df))
print(len(art_df))

31788324
1371980
105542


In [4]:
art_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   article_id                    105542 non-null  int64 
 1   product_code                  105542 non-null  int64 
 2   prod_name                     105542 non-null  object
 3   product_type_no               105542 non-null  int64 
 4   product_type_name             105542 non-null  object
 5   product_group_name            105542 non-null  object
 6   graphical_appearance_no       105542 non-null  int64 
 7   graphical_appearance_name     105542 non-null  object
 8   colour_group_code             105542 non-null  int64 
 9   colour_group_name             105542 non-null  object
 10  perceived_colour_value_id     105542 non-null  int64 
 11  perceived_colour_value_name   105542 non-null  object
 12  perceived_colour_master_id    105542 non-null  int64 
 13 

In [5]:
# transaction & customer 중복 key 컬럼 확인
print(set(transaction_pd.columns)&set(customer_df.columns))

# customer & transaction merge (key : customer id)
cust_tran_df = customer_df.merge(transaction_pd, how='inner', on = ['customer_id'])

# cust_tran_df.isna().sum()

{'customer_id'}


In [6]:
set(art_df.columns)&set(cust_tran_df.columns)
total_df = cust_tran_df.merge(art_df, how='inner', on =['article_id'])

In [7]:
# 패션 뉴스, 온라인 마케팅 등록 여부
total_df['FN'] = total_df['FN'].fillna(0)
total_df['Active'] = total_df['Active'].fillna(0)

In [9]:
# FN, ACtive 0 채우기 확인
# total_df.isna().sum()

### club_member_status 
- H&M 클럽 멤버십은 회원이 계정을 생성해 가입한 뒤, 구매·리뷰·기타 활동을 통해 포인트를 적립하고 이를 리워드로 교환하는 로열티 프로그램
- ACTIVE: 현재 클럽 멤버십이 활성화된 회원
- PRE-CREATE: 계정/멤버십이 사전 생성되었으나 완전 활성화 전 상태
- LEFT CLUB: 클럽을 탈퇴한 회원 (브랜드 완전이탈이 확정이 아님, 조금 애매함)
- NaN: 멤버십 상태 정보가 없는 결측값

**H&M club_member_status NaN

오프라인 구매만 한 고객,
멤버십 상태 수집 전 가입자,
데이터 병합 과정에서 상태 누락,
멤버십과 무관한 일반 고객

In [10]:
total_df['club_member_status'].unique()

array(['ACTIVE', nan, 'PRE-CREATE', 'LEFT CLUB'], dtype=object)

In [11]:
# club_member_status na값 삭제
# 62,000(NA) / 30,000,000 ≈ 0.21%
total_df['club_member_status'] = total_df['club_member_status'].replace('nan',np.nan)
total_df = total_df.dropna(subset=['club_member_status'])

In [ ]:
# total_df.isna().sum()

customer_id                          0
FN                                   0
Active                               0
club_member_status                   0
fashion_news_frequency          129103
age                             126598
postal_code                          0
t_dat                                0
article_id                           0
price                                0
sales_channel_id                     0
product_code                         0
prod_name                            0
product_type_no                      0
product_type_name                    0
product_group_name                   0
graphical_appearance_no              0
graphical_appearance_name            0
colour_group_code                    0
colour_group_name                    0
perceived_colour_value_id            0
perceived_colour_value_name          0
perceived_colour_master_id           0
perceived_colour_master_name         0
department_no                        0
department_name          

#### 패션 뉴스레터 수신 빈도 
- array(['NONE', 'Regularly', nan, 'Monthly'], dtype=object)

| 값         | 의미              |
| --------- | --------------- |
| NONE      | 수신 안 함 /뉴스레터 거부자와 정보 없음 고객을 같은 그룹|
| Monthly   | 월 1회 수신         |
| Regularly | 정기 수신 (월 1회 이상) |
| NaN       | 정보 없음           |


In [12]:
total_df['fashion_news_frequency'] = total_df['fashion_news_frequency'].fillna('UNKNOWN')
print(total_df['fashion_news_frequency'].unique())
total_df.isna().sum()

['NONE' 'Regularly' 'UNKNOWN' 'Monthly']


customer_id                          0
FN                                   0
Active                               0
club_member_status                   0
fashion_news_frequency               0
age                             126598
postal_code                          0
t_dat                                0
article_id                           0
price                                0
sales_channel_id                     0
product_code                         0
prod_name                            0
product_type_no                      0
product_type_name                    0
product_group_name                   0
graphical_appearance_no              0
graphical_appearance_name            0
colour_group_code                    0
colour_group_name                    0
perceived_colour_value_id            0
perceived_colour_value_name          0
perceived_colour_master_id           0
perceived_colour_master_name         0
department_no                        0
department_name          

### 나이 (결측치 알수없음으로 채움)
- min(10대), max(90대) - 이상치인지 확인 필요
 (16.0, 99.0)


In [13]:
min(total_df['age']), max(total_df['age'])

(16.0, 99.0)

In [14]:
total_df[total_df['age']>=90].value_counts() # 1393건

customer_id                                                       FN   Active  club_member_status  fashion_news_frequency  age   postal_code                                                       t_dat       article_id  price     sales_channel_id  product_code  prod_name            product_type_no  product_type_name  product_group_name  graphical_appearance_no  graphical_appearance_name  colour_group_code  colour_group_name  perceived_colour_value_id  perceived_colour_value_name  perceived_colour_master_id  perceived_colour_master_name  department_no  department_name     index_code  index_name          index_group_no  index_group_name  section_no  section_name                    garment_group_no  garment_group_name  detail_desc                                                                                                                                                                                    
bc870dbac4a150f6e5cf8e1d71e59d77d3da3f171dae2433b7d93dd3341e3b07  0.0  0.0     ACTI

In [15]:
bins = [0, 19, 29, 39, 49, 59, 69, 100]
labels = ['10대 미만', '10대', '20대', '30대', '40대', '50대', '60대 이상']
total_df['age_cut'] = pd.cut(total_df['age'], bins=bins, labels=labels, right=True).astype('str')
type(total_df['age_cut'])
total_df['age_cut'] = total_df['age_cut'].str.replace('nan', 'UNKNOWN')


In [ ]:
total_df['age_cut'].unique()

array(['30대', '10대', '40대', '20대', '60대 이상', '50대', '10대 미만', 'UNKNOWN'],
      dtype=object)

In [ ]:
total_df['age_cut'].value_counts()

age_cut
10대        13039351
20대         6416888
40대         5130097
30대         4901185
50대         1201949
10대 미만       690192
60대 이상       219899
UNKNOWN      126598
Name: count, dtype: int64

In [ ]:
total_df.columns

Index(['customer_id', 'FN', 'Active', 'club_member_status',
       'fashion_news_frequency', 'age', 'postal_code', 't_dat', 'article_id',
       'price', 'sales_channel_id', 'product_code', 'prod_name',
       'product_type_no', 'product_type_name', 'product_group_name',
       'graphical_appearance_no', 'graphical_appearance_name',
       'colour_group_code', 'colour_group_name', 'perceived_colour_value_id',
       'perceived_colour_value_name', 'perceived_colour_master_id',
       'perceived_colour_master_name', 'department_no', 'department_name',
       'index_code', 'index_name', 'index_group_no', 'index_group_name',
       'section_no', 'section_name', 'garment_group_no', 'garment_group_name',
       'detail_desc', 'age_cut'],
      dtype='object')

### 1차 컬럼 분류

In [40]:
total_df = total_df[['customer_id', 'FN', 'club_member_status',
       'fashion_news_frequency', 'age', 't_dat', 'price', 'product_group_name','age_cut']]
EDA_columns = total_df

In [41]:
EDA_columns.head(5)

,customer_id,FN,club_member_status,fashion_news_frequency,age,t_dat,price,product_group_name,age_cut
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.044051,Garment Upper body,30대
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.035576,Garment Upper body,30대
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2018-12-27,0.030492,Garment Upper body,30대
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-02,0.010153,Garment Full body,30대
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49.0,2019-05-25,0.050831,Garment Upper body,30대


### price 값 수치 의미 (기업 미공개- 정규화된 데이터)

| 값 예시                          | 해석                      |                 |
| -------------------------- | ----------------------- | --------------- |
| `0.009`                    | 원본에서 매우 낮은 가격 → 스케일링 결과 |                 |
| `0.0278` (dataset average) | 스케일링된 평균 거래 가격          | ([ARAMDAUN][1]) |
| `0.5915` (max)             | 스케일된 가장 비싼 상품           | ([ARAMDAUN][1]) |

[1]: https://ars420.tistory.com/42?utm_source=chatgpt.com "[Kaggle] H&M Personalized Fashion Recommendations"


In [42]:
type(EDA_columns['t_dat'])

pandas.core.series.Series

In [47]:
# 평균 구매 간격
EDA_columns = EDA_columns.sort_values(['customer_id','t_dat'])
EDA_columns['prev_date'] = EDA_columns.groupby('customer_id')['t_dat'].shift(1)
EDA_columns['gap'] = (EDA_columns['t_dat'] - EDA_columns['prev_date']).dt.days
EDA_columns = EDA_columns.fillna(0) # 시작날은 시차 0 


In [ ]:
# 고객 구매 주기에 따른 평균 일 수
churn_label = EDA_columns.groupby(['customer_id'])['gap'].mean().astype(int).reset_index()
churn_label['churn'] = np.where(churn_label['gap']>0,0,1)
churn_label



In [ ]:
# 이탈 라벨링 데이터를 EDA df에 merge (0:미이탈 / 1:이탈)
churn_label_merge = churn_label[['customer_id','churn']]
churn_label_merge = churn_label_merge.merge(EDA_columns,on=['customer_id'],how='inner' )
churn_label_merge.head(20)

In [61]:
# 이탈에 대한 gap 확인
churn_label_merge[churn_label_merge['churn']==1]

,customer_id,churn,FN,club_member_status,fashion_news_frequency,age,t_dat,price,product_group_name,age_cut,prev_date,gap
125,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,1,0.0,ACTIVE,NONE,54.0,2019-06-09,0.030492,Underwear,40대,0,0.0
126,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,1,0.0,ACTIVE,NONE,54.0,2019-06-09,0.030492,Underwear,40대,2019-06-09 00:00:00,0.0
266,00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77...,1,0.0,ACTIVE,NONE,20.0,2020-01-05,0.022864,Garment Upper body,10대,0,0.0
267,00007e8d4e54114b5b2a9b51586325a8d0fa74ea23ef77...,1,0.0,ACTIVE,NONE,20.0,2020-01-05,0.030492,Garment Lower body,10대,2020-01-05 00:00:00,0.0
268,00008469a21b50b3d147c97135e25b4201a8c58997f787...,1,0.0,ACTIVE,NONE,20.0,2018-11-12,0.016932,Garment Upper body,10대,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
31725967,ffffa28cd7ab5d1cbbbfe7b582b1c419270cc0539f3dae...,1,1.0,ACTIVE,Regularly,22.0,2018-10-10,0.033898,Garment Lower body,10대,0,0.0
31725968,ffffa28cd7ab5d1cbbbfe7b582b1c419270cc0539f3dae...,1,1.0,ACTIVE,Regularly,22.0,2018-10-10,0.027102,Garment Lower body,10대,2018-10-10 00:00:00,0.0
31725969,ffffa28cd7ab5d1cbbbfe7b582b1c419270cc0539f3dae...,1,1.0,ACTIVE,Regularly,22.0,2018-10-10,0.027102,Garment Lower body,10대,2018-10-10 00:00:00,0.0
31725970,ffffaff3905b803d1c7e153a1378a5151e1f34f236ba54...,1,1.0,ACTIVE,Regularly,21.0,2018-10-26,0.122017,Garment Upper body,10대,0,0.0


In [ ]:
col_kr_map = {
    'customer_id': '고객ID',
    'churn': '이탈여부',  # 0: 잔존, 1: 이탈
    'FN': '패션뉴스구독 여부',  # 패션뉴스구독 여부
    'club_member_status': '멤버십상태',  # ACTIVE, PRE-CREATE, LEFT CLUB
    'fashion_news_frequency': '뉴스레터수신빈도',  # NONE, Monthly, Regularly
    'age': '연령',
    't_dat': '구매일',  # 거래/주문 날짜
    'price': '구매금액',
    'product_group_name': '상품그룹',
    'age_cut': '연령대',
    'prev_date': '이전구매일',  # 이전 구매 날짜
    'gap': '구매간격(일)'  # 현재-이전 구매일 차이
}

# 컬럼명 변경
churn_label_merge.rename(columns=col_kr_map, inplace=True)

# 확인
churn_label_merge.head()

,고객ID,이탈여부,최근구매횟수,멤버십상태,뉴스레터수신빈도,연령,구매일,구매금액,상품그룹,연령대,이전구매일,구매간격(일)
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0.0,ACTIVE,NONE,49.0,2018-12-27,0.044051,Garment Upper body,30대,0,0.0
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0.0,ACTIVE,NONE,49.0,2018-12-27,0.035576,Garment Upper body,30대,2018-12-27 00:00:00,0.0
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0.0,ACTIVE,NONE,49.0,2018-12-27,0.030492,Garment Upper body,30대,2018-12-27 00:00:00,0.0
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0.0,ACTIVE,NONE,49.0,2019-05-02,0.010153,Garment Full body,30대,2018-12-27 00:00:00,126.0
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0.0,ACTIVE,NONE,49.0,2019-05-25,0.050831,Garment Upper body,30대,2019-05-02 00:00:00,23.0


In [ ]:
churn_label_merge.to_csv('C:/Users/Playdata/Downloads/H_M_EDA_df.csv',drop=True)